# 跨会话记忆（Cross-Session Memory）

> *把智能体状态保存在多个独立会话之间，让回访的用户获得连续性，而不是一块白板。*

想象一个同事一夜之间失忆了。每天早上，你都得重新解释你的项目、你的偏好、昨天的决定。没有跨会话记忆的 AI 智能体就是这个感觉。最有用的助手，是那些能随时间学习和记忆的助手。

没有跨会话记忆，每段对话都从零开始。一个花了三十分钟教智能体做事的用户，第二天回来面对的是一块白板。这不仅浪费用户的时间，也限制了智能体能提供的价值。

跨会话记忆通过在会话之间加一层持久化来解决这个问题。「持久化」指保存数据，让它在程序停止运行后依然存在。会话结束时，智能体保存自己的相关状态，包括抽取的事实、用户偏好、对话摘要和任务上下文。系统把这些状态序列化（转换成可存储的格式），写入持久化的后端。新会话开始时，系统识别出回访用户，加载他的记忆快照，并在第一个轮次之前用这些上下文初始化智能体。最终得到的，是一个能在数天、数周、数月之间「记得」用户的智能体。

工程上的挑战很实际，也很重要。你要选一个序列化格式：JSON 人类可读，Pickle 能处理复杂的 Python 对象。你要选一个存储后端：Redis 快，SQLite 搭建简单，S3 扩展性好。你还要处理没有历史记忆时的冷启动场景。当存储的历史超出上下文窗口时，你还得决定加载多少历史上下文。

**在本 notebook 中你将构建：**
1. 一个 `SessionState` 数据类，承载可序列化的智能体状态。
2. 一个 `StorageBackend`，附带 SQLite 实现。
3. 一个 `CrossSessionManager`，管理会话生命周期（保存、加载、冷启动）。
4. 一个 `CrossSessionAgent`，把持久化记忆接入对话循环。
5. 一个可运行的演示：同一用户跨越多个会话。


## 关键概念

- **会话序列化（session serialization）**：把智能体的内存状态转换成可存储的格式。状态包括对话历史、抽取的事实、用户偏好和任务上下文。JSON 人类可读、便于检查。Pickle 能处理任意 Python 对象。Protobuf 提供紧凑的二进制编码，并支持模式演进（schema evolution，即在不破坏旧数据的前提下修改数据格式）。

- **状态持久化后端（persistence backend）**：序列化状态在会话之间存放的持久存储。Redis 读取低于一毫秒，适合低延迟的会话恢复。SQLite 零依赖，提供本地持久化。S3/GCS 持久性高，能扩展到百万级用户。PostgreSQL 支持对存储状态做复杂查询。

- **会话恢复（session resumption）**：识别回访用户，加载其存储的记忆快照。系统在第一个对话轮次之前恢复（hydrate）智能体的内部状态。这一步必须足够快，避免会话开始时出现可感知的延迟。

- **用户识别（user identification）**：把进来的请求映射到一个持久的用户身份（用户 ID、API key、会话 token 或认证声明）。智能体必须取到正确的记忆分区。严格的隔离能防止记忆在用户之间泄漏。

- **记忆加载策略（loading strategy）**：当存储的历史超出上下文窗口时，系统必须选择加载什么。选项包括：完整历史、最近 N 个轮次、仅摘要，或按相关性排序的检索。按相关性排序指查询存储的记忆，找出与当前对话最相关的内容。

- **冷启动处理（cold start）**：当用户没有任何历史会话时，初始化智能体状态。包括设置默认偏好，建立基线上下文。


## 架构

跨会话记忆工作在会话的边界上。会话结束时序列化状态，新会话开始时反序列化。一个持久化的存储后端填补两者之间的空隙。

<p align="center">
  <img src="../../images/diagrams/21_cross_session_memory.svg" alt="架构图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart LR
    subgraph "Session End"
        AS[Agent State] --> SER[Serializer<br/>JSON / pickle]
    end

    SER --> SB[(Storage Backend<br/>Redis / SQLite / S3)]

    subgraph "New Session"
        REQ[Incoming Request] --> UID[User<br/>Identifier]
        UID -->|user_id| ML[Memory<br/>Loader]
        SB -->|stored state| ML
        ML --> LS{Loading<br/>Strategy}
        LS -->|full| CTX1[Full History]
        LS -->|partial| CTX2[Last-N Turns]
        LS -->|summary| CTX3[Summary Only]
        CTX1 --> AC[Agent Context]
        CTX2 --> AC
        CTX3 --> AC
    end

    subgraph "Cold Start"
        UID -->|no prior state| CS[Cold Start<br/>Handler]
        CS -->|defaults +<br/>onboarding| AC
    end

    style SB fill:#4a9eff,color:#fff
    style UID fill:#845ef7,color:#fff
    style LS fill:#ffa94d,color:#fff
    style CS fill:#ff6b6b,color:#fff
```
</details>

**数据流：** 会话结束时，智能体状态经过序列化器（Serializer，JSON 或 pickle）。序列化后的状态写入存储后端（Storage Backend），以用户 ID 为键。新会话开始时，用户标识（User Identifier）从来进请求中提取用户身份。记忆加载器（Memory Loader）向存储后端查询该用户的存储状态。加载策略（Loading Strategy）根据历史大小和上下文窗口上限决定加载多少。如果不存在历史状态，冷启动处理器（Cold Start Handler）初始化默认值。加载的状态随后注入智能体上下文（Agent Context）。


## 准备工作

安装依赖并配置 API 访问。


In [ ]:
%pip install -q python-dotenv deepagents langchain-openai

导入 OpenAI SDK 和标准库工具。API key 从 `.env` 文件加载。


In [ ]:
import os
import json
import sqlite3
from datetime import datetime
from dataclasses import dataclass, asdict
from abc import ABC, abstractmethod
from typing import Optional

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 实现

我们分四部分构建跨会话记忆：

1. 一个 `SessionState` 数据类，保存智能体需要记住的一切。
2. 一个 `StorageBackend`，把状态持久化到 SQLite。
3. 一个 `CrossSessionManager`，负责保存、加载和冷启动。
4. 一个 `CrossSessionAgent`，把记忆接入对话循环。


### 会话状态

第一步是定义*保存什么*。我们捕获五样东西：对话摘要、抽取的事实、用户偏好、最近的消息，以及会话元数据。


In [ ]:
@dataclass
class SessionState:
    """在会话之间持久化的智能体状态快照。"""

    user_id: str
    conversation_summary: str
    extracted_facts: list[str]
    user_preferences: dict
    last_n_messages: list[dict]
    session_count: int = 0
    last_active: str = ""

    def to_dict(self) -> dict:
        """转换成可 JSON 序列化的字典。"""
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict) -> "SessionState":
        """从字典重建一个 SessionState。"""
        return cls(**data)

### 存储后端

把存储后端想象成一个文件柜。每个抽屉用用户 ID 标注。每个抽屉里放一个文件夹，装着该用户的会话状态。后端的职责就是归档和取出这些文件夹。

我们先定义一个抽象接口，让你可以在不改动其余代码的情况下更换后端。然后用 SQLite（一个存在单个文件里的轻量级数据库）构建一个具体后端。


In [ ]:
class StorageBackend(ABC):
    """会话状态持久化接口。"""

    @abstractmethod
    def save(self, user_id: str, state: SessionState) -> None: ...

    @abstractmethod
    def load(self, user_id: str) -> Optional[SessionState]: ...

    @abstractmethod
    def delete(self, user_id: str) -> None: ...

    @abstractmethod
    def list_users(self) -> list[str]: ...


class SQLiteBackend(StorageBackend):
    """把会话状态持久化到 SQLite 数据库。"""

    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS session_state (
                user_id    TEXT PRIMARY KEY,
                state_json TEXT NOT NULL,
                updated_at TEXT NOT NULL
            )
        """)
        self.conn.commit()

    def save(self, user_id: str, state: SessionState) -> None:
        self.conn.execute(
            """INSERT OR REPLACE INTO session_state
               (user_id, state_json, updated_at) VALUES (?, ?, ?)""",
            (user_id, json.dumps(state.to_dict()), datetime.now().isoformat()),
        )
        self.conn.commit()

    def load(self, user_id: str) -> Optional[SessionState]:
        row = self.conn.execute(
            "SELECT state_json FROM session_state WHERE user_id = ?",
            (user_id,),
        ).fetchone()
        if row is None:
            return None
        return SessionState.from_dict(json.loads(row[0]))

    def delete(self, user_id: str) -> None:
        self.conn.execute(
            "DELETE FROM session_state WHERE user_id = ?", (user_id,),
        )
        self.conn.commit()

    def list_users(self) -> list[str]:
        rows = self.conn.execute("SELECT user_id FROM session_state").fetchall()
        return [r[0] for r in rows]

### 用 LLM 抽取事实并生成摘要

每个会话结束时，我们让 LLM 抽取关于用户的关键事实，并把对话压缩成一段简短摘要。这些紧凑的表示把最重要的信息带进下一个会话，不用重放每条消息。


In [ ]:
def extract_facts(messages: list[dict], llm_client: OpenAI) -> list[str]:
    """让 LLM 从对话消息中抽取用户的关键事实。"""
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
    )
    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "Extract key facts about the user from this conversation. "
                    "Return each fact on its own line, prefixed with '- '. "
                    "Focus on: name, role, projects, preferences, and goals."
                ),
            },
            {"role": "user", "content": transcript},
        ])
    raw = to_text(response.content)
    return [
        line.strip("- ").strip()
        for line in raw.strip().split("\n")
        if line.strip()
    ]


def summarize_conversation(messages: list[dict], llm_client: OpenAI) -> str:
    """把对话历史压缩成简短摘要。"""
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
    )
    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "Summarize this conversation in 2-3 sentences. "
                    "Capture the main topics and any decisions made."
                ),
            },
            {"role": "user", "content": transcript},
        ])
    return to_text(response.content).strip()

### 跨会话管理器

管理器位于智能体和存储后端之间。它承担三项职责：

1. **恢复**会话：加载存储的状态，递增会话计数器，应用加载策略。
2. **结束**会话：序列化当前状态并写入存储。
3. **冷启动**：当用户没有历史记录时，创建默认状态。


In [ ]:
class CrossSessionManager:
    """协调会话持久化、恢复和冷启动。"""

    def __init__(
        self,
        backend: StorageBackend,
        loading_strategy: str = "full",
        max_messages: int = 20,
    ):
        self.backend = backend
        self.loading_strategy = loading_strategy
        self.max_messages = max_messages  # 供 "last_n" 策略使用

    def resume_session(self, user_id: str) -> SessionState:
        """加载历史状态，或为新用户冷启动。"""
        state = self.backend.load(user_id)
        if state is None:
            print(f"[Cold start] No prior state for '{user_id}'. Creating fresh state.")
            return self._cold_start(user_id)

        state.session_count += 1
        state.last_active = datetime.now().isoformat()
        loaded = self._apply_loading_strategy(state)
        print(
            f"[Resumed] User '{user_id}', session #{loaded.session_count}. "
            f"Loaded {len(loaded.last_n_messages)} messages, "
            f"{len(loaded.extracted_facts)} facts."
        )
        return loaded

    def end_session(self, user_id: str, state: SessionState) -> None:
        """把当前状态持久化到后端。"""
        state.last_active = datetime.now().isoformat()
        self.backend.save(user_id, state)
        print(
            f"[Saved] User '{user_id}': "
            f"{len(state.last_n_messages)} messages, "
            f"{len(state.extracted_facts)} facts persisted."
        )

接下来我们在 `CrossSessionManager` 上定义两个辅助方法。
`_cold_start` 为首次使用的用户创建默认状态。
`_apply_loading_strategy` 按选定的策略（完整、最近 N 条、仅摘要）裁剪加载的状态。


In [ ]:
def __cold_start(self, user_id: str) -> SessionState:
    """为首次使用的用户初始化默认状态。"""
    return SessionState(
        user_id=user_id,
        conversation_summary="",
        extracted_facts=[],
        user_preferences={},
        last_n_messages=[],
        session_count=1,
        last_active=datetime.now().isoformat(),
    )

def __apply_loading_strategy(self, state: SessionState) -> SessionState:
    """按选定的策略裁剪加载的状态。"""
    if self.loading_strategy == "full":
        return state
    elif self.loading_strategy == "last_n":
        # 只保留最近的消息
        state.last_n_messages = state.last_n_messages[-self.max_messages :]
        return state
    elif self.loading_strategy == "summary":
        # 完全丢弃消息，只依赖摘要 + 事实
        state.last_n_messages = []
        return state
    return state

CrossSessionManager._cold_start = __cold_start

CrossSessionManager._apply_loading_strategy = __apply_loading_strategy


### 组装起来：智能体

智能体把跨会话记忆接入 LLM 对话循环。每个轮次，它用存储的摘要、抽取的事实和会话元数据构建系统提示词。这样，模型在读到新消息之前，就已经掌握了用户的上下文。


In [ ]:
class CrossSessionAgent:
    """记忆可跨会话持久化的聊天智能体。"""

    def __init__(self, manager: CrossSessionManager, model):
        self.manager = manager
        self.model = model
        self.state: Optional[SessionState] = None
        self.user_id: Optional[str] = None

    def start_session(self, user_id: str) -> None:
        """为指定用户恢复会话，或冷启动新会话。"""
        self.user_id = user_id
        self.state = self.manager.resume_session(user_id)

    def chat(self, user_input: str) -> str:
        """发送消息，带上跨会话上下文获取回复。"""
        self.state.last_n_messages.append(
            {"role": "user", "content": user_input}
        )

        # 用持久化记忆构建系统提示词
        system_parts = [
            "You are a helpful assistant with memory across sessions.",
            "Keep your replies concise (2-3 sentences).",
        ]
        if self.state.conversation_summary:
            system_parts.append(
                f"\nPrevious conversation summary:\n{self.state.conversation_summary}"
            )
        if self.state.extracted_facts:
            facts_str = "\n".join(f"- {f}" for f in self.state.extracted_facts)
            system_parts.append(f"\nKnown facts about this user:\n{facts_str}")
        if self.state.session_count > 1:
            system_parts.append(
                f"\nThis is session #{self.state.session_count} with this user."
            )

        # 跨会话记忆每轮变化，系统提示词每轮都不同，所以每轮重建智能体
        agent = create_deep_agent(
            model=self.model,
            system_prompt="\n".join(system_parts),
        )
        result = agent.invoke({"messages": list(self.state.last_n_messages)})

        reply = to_text(result["messages"][-1].content)
        self.state.last_n_messages.append(
            {"role": "assistant", "content": reply}
        )
        return reply

会话结束时，智能体抽取事实并总结对话。
然后它通过管理器把这些状态持久化。
这是实时对话和持久存储之间的桥梁。


In [ ]:
def _end_session(self) -> None:
    """抽取事实、生成摘要并持久化状态。"""
    if self.state and len(self.state.last_n_messages) > 0:
        self.state.extracted_facts = extract_facts(self.state.last_n_messages)
        self.state.conversation_summary = summarize_conversation(self.state.last_n_messages)
    self.manager.end_session(self.user_id, self.state)

CrossSessionAgent.end_session = _end_session


## 运行示例

我们模拟同一用户的两个会话，然后演示一个新用户的冷启动。这展示了跨会话记忆如何创造连续性。


### 会话 1：初次见面

Alice 第一次和智能体聊天。因为不存在历史状态，管理器触发冷启动。最后，我们保存她的会话。


In [ ]:
backend = SQLiteBackend()  # 演示用内存 SQLite
manager = CrossSessionManager(backend, loading_strategy="full")
agent = CrossSessionAgent(manager, model=llm)

agent.start_session("alice_123")

print("=" * 50)
print("SESSION 1: First Meeting")
print("=" * 50, "\n")

session_1_messages = [
    "Hi! I'm Alice. I'm a data scientist at Acme Corp.",
    "I'm building a recommendation system with collaborative filtering.",
    "I prefer Python and use pandas for most of my data work.",
]

for msg in session_1_messages:
    print(f"User:  {msg}")
    reply = agent.chat(msg)
    print(f"Agent: {reply}\n")

agent.end_session()

### 会话 2：智能体记得

Alice 稍后回来。智能体加载她存储的摘要和事实。不需要她重复，就应该记得她的名字、职位和项目。


In [ ]:
agent_s2 = CrossSessionAgent(manager, model=llm)
agent_s2.start_session("alice_123")

print("=" * 50)
print("SESSION 2: Alice Returns")
print("=" * 50, "\n")

session_2_messages = [
    "Hey, I'm back! What do you remember about me?",
    "Any suggestions for improving my recommendation system?",
]

for msg in session_2_messages:
    print(f"User:  {msg}")
    reply = agent_s2.chat(msg)
    print(f"Agent: {reply}\n")

agent_s2.end_session()

### 冷启动：新用户

Bob 没有任何历史记录就来了。管理器创建默认状态，智能体从零开始。


In [ ]:
agent_new = CrossSessionAgent(manager, model=llm)
agent_new.start_session("bob_456")

print("=" * 50)
print("NEW USER: Cold Start")
print("=" * 50, "\n")

msg = "Hello! Can you help me with a machine learning project?"
print(f"User:  {msg}")
reply = agent_new.chat(msg)
print(f"Agent: {reply}\n")

agent_new.end_session()

### 检查存储的状态

我们看看两个会话之后，后端为 Alice 保存了什么。你会看到抽取的事实、对话摘要和会话元数据。


In [ ]:
alice_state = backend.load("alice_123")

print("Stored State for alice_123")
print("-" * 40)
print(f"Session count: {alice_state.session_count}")
print(f"Last active:   {alice_state.last_active}")
print(f"\nExtracted facts:")
for fact in alice_state.extracted_facts:
    print(f"  - {fact}")
print(f"\nConversation summary:")
print(f"  {alice_state.conversation_summary}")
print(f"\nStored messages: {len(alice_state.last_n_messages)}")
print(f"All users in backend: {backend.list_users()}")

### 比较加载策略

当存储的历史变大，你需要选择加载什么。这里我们用 Alice 保存的状态比较三种策略：

- **full**：全部加载。适合历史较短的用户。
- **last_n**：只保留最近 N 条消息。裁掉旧上下文。
- **summary**：丢弃所有消息。只依赖摘要和抽取的事实。


In [ ]:
print("Loading Strategy Comparison (alice_123)")
print("=" * 50, "\n")

for strategy in ["full", "last_n", "summary"]:
    test_manager = CrossSessionManager(
        backend, loading_strategy=strategy, max_messages=4
    )
    loaded = test_manager.resume_session("alice_123")
    print(f"Strategy: {strategy}")
    print(f"  Messages loaded:  {len(loaded.last_n_messages)}")
    print(f"  Facts available:  {len(loaded.extracted_facts)}")
    has_summary = "yes" if loaded.conversation_summary else "no"
    print(f"  Summary present:  {has_summary}")
    print()

清理内存数据库。


In [ ]:
backend.conn.close()

## 权衡

### 跨会话记忆适合的场景

- **个性化助手。** 定期回访的用户获得更好的体验。智能体记得他们的名字、偏好和进行中的项目。
- **长周期任务。** 一个持续多天的研究项目可以从中断处继续。智能体追踪哪些已经完成、哪些还没做。
- **减少用户的重复劳动。** 用户不必重复交代背景。这节省时间，也减少挫败感。

### 什么时候会失效

- **隐私和数据保留。** 跨会话存储用户数据带来合规问题（GDPR、CCPA）。你需要明确的策略：数据保留多久、用户如何删除。
- **陈旧的记忆。** 几个月前抽取的事实可能已不再成立。用户可能换了工作或项目。没有让旧事实过期的机制，智能体会做出错误假设。
- **规模化后的存储成本。** 用户达到百万级时，存储后端成为真正的基建问题。你要规划数据库容量、备份和访问模式。
- **上下文窗口限制。** 即便有加载策略，多个会话累积的状态也可能超出单次提示词的容量。你需要把这项技术与摘要或基于检索的方法结合。


## 延伸阅读

- Packer, C., et al. (2023). ["MemGPT: Towards LLMs as Operating Systems."](https://arxiv.org/abs/2310.08560) 提出 LLM 的虚拟内存分页。智能体在有限的上下文窗口内工作，但从持久化的外部存储分页调入数据。

- [LangChain Memory 文档](https://python.langchain.com/docs/modules/memory/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：介绍对话记忆类型（缓冲、摘要、实体），内置对文件和数据库的持久化。

- [Letta (MemGPT) 持久化模型](https://docs.letta.com?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：说明 Letta 在会话之间持久化智能体状态的方法，包括记忆层级和后端架构。

- [SQLite: 何时使用](https://www.sqlite.org/whentouse.html?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：指导什么时候 SQLite 是合适的数据库选择。很多智能体原型用 SQLite 做本地状态持久化，所以这份指南很相关。

- [Redis 持久化文档](https://redis.io/docs/latest/operate/oss_and_stack/management/persistence/)：解释 Redis 的持久性选项（RDB 快照、AOF 日志）。当你需要在生产环境规模下实现亚毫秒级会话恢复时可以参考。


---

*← 上一篇：[20：记忆检索模式](../20_memory_retrieval_patterns/) · 下一篇：[22：多智能体共享记忆](../22_multi_agent_shared_memory/) →*


## 🧪 动手试一试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：状态版本管理
给 `SessionState` 加一个 `version` 整数字段。`CrossSessionManager` 每次保存状态时递增版本号，并把上一个版本存进一张 `history` 表。实现一个 `rollback(version)` 方法，恢复到之前的状态。测试方法：保存 3 个版本，然后回滚到第一个。

### 挑战 2：存储增长追踪
对话中每过 5 个轮次，就把 `SessionState` 序列化成 JSON 并记录字节数。跑一段 30 个轮次的对话，画出轮次编号与状态大小的关系图。找出哪个组成部分（消息、事实还是偏好）增长最快。

### 挑战 3：多用户会话管理器
扩展 `CrossSessionManager`，让它接受一个 `user_id` 参数。把每个用户的状态存在单独的 SQLite 行（或文件）里。创建两个用户，各跑 5 个轮次，保存，然后分别恢复两个用户。验证他们的状态相互隔离。这是对 22 多智能体共享记忆中多智能体模式的补充。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--21-cross-session-memory--cross-session-memory)